# NLP4 - Topic Modeling: NMF and LDA (Student Version)


In this session we will explore two techniques for discovering themes in text: Non-negative Matrix Factorization (NMF) and Latent Dirichlet Allocation (LDA). We begin with preprocessing, build an NMF model using TF-IDF features, and then use token counts to train and interpret an LDA model.

NMF represents a non-negative document-term matrix as document-topic and topic-term weights. LDA takes a probabilistic approach, representing documents as mixtures of topics and topics as distributions over words. The lab walks through both methods with discussion, step-by-step code, and interpretation.

## Student instructions

Complete the code cells marked with `TODO`. Run the notebook from top to bottom, inspect the outputs, and write your interpretations in the discussion prompts. The solution notebook contains the completed code.

---

## Install Libraries

In [ ]:
#first lets install datasets library
# !pip install datasets
# !python -m spacy download pt_core_news_lg

## Imports

In [ ]:
from itertools import count

#dataset library
from datasets import load_dataset

#load dataset
dataset = load_dataset("tclopess/sinopsys_movies_portuguese")
#convert it to pandas and slice the first 3000 data points
df_sinop = dataset['train'].to_pandas()[:3000]


#NLP tool box nltk
import nltk
from nltk.corpus import stopwords
#getting stop words
# nltk.download('stopwords')
stop = list(set(stopwords.words('portuguese')))
print(stop)

#string library
import string
#get list of punctuations
pontuacoes = string.punctuation
print(pontuacoes)

#NLP toolbox spacy
import spacy
#load portuguese module large
nlp = spacy.load("pt_core_news_lg")

#other python support libraries and methods
import itertools
from collections import Counter
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#dataframes library
import pandas as pd

#LDA library
import gensim
import gensim.corpora as corpora
from gensim.models.ldamodel import LdaModel

### Practicing


1 - Using the concepts from the previous class, create a function that takes a string as a parameter and returns a list of pre-processed tokens. The tokens should be lowercase, lemmas, and must not be punctuation or stopwords.

In [ ]:
# TODO: complete the preprocessing function.
# It should return lowercase lemmas and remove stopwords and punctuation.
def preprocessing(text):
    # TODO: process text with nlp and return the filtered lemmas
    pass


2 - Using the function you created in the previous exercise, preprocess all the synopsis texts contained in the dataframe.

In [ ]:
#exemplo das sinopses
df_sinop.head()

In [ ]:
# TODO: apply preprocessing() to every synopsis in df_sinop['sinopse']
preprocessed_docs = ...


In [ ]:
#exemplo de uma sinopse preprocessada
preprocessed_docs[0]

In [ ]:
# TODO: create a tokens column by applying preprocessing() to each synopsis


## Non-negative Matrix Factorization (NMF)

NMF is a matrix factorization method. Given a non-negative document-term matrix **X**, it approximates the matrix as **W × H**. Here, rows of **W** describe documents in terms of topics, while rows of **H** describe topics in terms of words. With text, TF-IDF is a common input: each document is represented by the importance of its terms.

Unlike LDA, NMF is not a probabilistic generative model. Its topic weights are non-negative and useful for ranking, but they are not automatically probabilities that sum to one. Topic labels still require human interpretation.

### 1 — Prepare a TF-IDF matrix from the tokens

The corpus has already been lowercased, lemmatized, and filtered for stopwords above. Join each token list into a string, then let `TfidfVectorizer` build a document-term matrix. `min_df=5` removes terms found in fewer than five documents, reducing noise and vocabulary size. What are the rows and columns of this matrix?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: join the tokens in each document into one string
texts_for_nmf = ...

# TODO: create a TfidfVectorizer with min_df=5 and max_df=0.90
# TODO: fit it to texts_for_nmf and create X_tfidf
# TODO: retrieve the feature names

# TODO: print the matrix shape and number of non-zero values in its first row


### 2 — Choose the number of topics and fit NMF

Set `n_components` to the number of topics to discover. There is no universally correct value: compare several values, inspect whether topics are coherent and distinct, and use domain knowledge. We start with 10 to make the result comparable to the LDA example. `random_state` makes initialization reproducible; `max_iter` sets the iteration limit.

In [ ]:
#definindo o K com base na quantidade de generos de filmes que temos no dataset
#converter a listas em formato string para lista
df_sinop['generos'] = df_sinop['generos'].apply(lambda x: eval(x) if isinstance(x, str) else x)
#fazendo um flatten na lista de generos
all_genres = list(itertools.chain.from_iterable(df_sinop['generos'].to_list()))
#contando a quantidade de generos unicos
unique_genres = set(all_genres)
print("Unique genres:", unique_genres)
len(unique_genres)


In [ ]:
from sklearn.decomposition import NMF

# TODO: choose the number of topics
n_topics = ...

# TODO: create an NMF model with n_components=n_topics, init="nndsvda",
# random_state=100, and max_iter=400
nmf_model = ...

# TODO: fit the model to X_tfidf and save the document-topic matrix as W
# TODO: save the topic-term matrix as H
# TODO: print W.shape, H.shape, and the reconstruction error


### 3 — Inspect the most important words in each topic

For each row in **H**, sort term weights from largest to smallest. The top words help us assign a human-readable label to the topic. Are the words related, and do two topics appear to describe the same theme?

In [ ]:
# TODO: implement a function that prints the top terms for each NMF topic.
# Hint: use model.components_, argsort(), and feature_names.


In [ ]:
# TODO: inspect NMF topic word lists and note which topics may relate to romance.


### 4 — Rank topics for each document

Use the corresponding row of **W** to see which topics are strongest in a synopsis. These are weights, not probabilities. We can normalize a row to sum to one for a relative comparison within that document; this normalization is a convenient display choice, not a probabilistic guarantee from NMF.

In [ ]:
# TODO: choose a document, get its row from W, normalize its topic weights
# for relative comparison, sort topics, and print the top three with weights.


### 5 — Find representative documents for a topic

A topic’s top words give one view; documents with the largest topic weights give another. Inspecting both helps validate our interpretation and spot topics that mix unrelated themes.

In [ ]:
# TODO: choose a topic and print the five documents with the highest weight
# for that topic in W. Include each document's synopsis.


**Discussion / solution.** Compare these synopses with the topic’s top words. If they share a recognizable subject, the interpretation is supported. If not, the topic may be broad, driven by common vocabulary, or one of several themes in the same documents. Topic modeling is exploratory: inspect examples before naming or using topics.

In [ ]:
# TODO: inspect representative documents for a topic and compare them with
# the topic's top words. Record whether the topic interpretation makes sense.


In [ ]:
# TODO: compare selected NMF topics with the movie genres of their top documents.


## Topic Modeling with Latent Dirichlet Allocation (LDA)

NMF showed how a non-negative matrix can be represented through document-topic and topic-term weights. LDA approaches topic modeling probabilistically: it represents each document as a mixture of topics, and each topic as a distribution over words. We will build the inputs, train a model, inspect both distributions, discuss the results, and finish with an interactive pyLDAvis visualization.

LDA uses token counts rather than TF-IDF weights. We will reuse the same preprocessed tokens so the comparison with NMF is meaningful.

### 1 — Think about the LDA model

Before coding, discuss the central idea of the paper that introduced Latent Dirichlet Allocation: a document can contain several topics, and each topic can generate words with different probabilities. In your own words, describe how this differs from assigning each document to exactly one topic.

### 2 — Build a word dictionary

Gensim needs a mapping between token strings and integer IDs. `corpora.Dictionary` builds this vocabulary from the preprocessed documents. This uses the same preprocessing as the TF-IDF and NMF sections: lowercase lemmas with stopwords and punctuation removed. Inspect a few ID-to-word mappings and the vocabulary size.

In [ ]:
# TODO: create a Gensim Dictionary from preprocessed_docs.
id2word = ...

# TODO: print the vocabulary size and inspect a few ID-token pairs.


### 3 — Convert documents into bag-of-words counts

LDA models word occurrences, so convert each tokenized synopsis to a bag-of-words representation with `doc2bow`. Each document becomes a list of `(token_id, count)` pairs. Word order is not retained. Inspect the first document and map its IDs back to words.

In [ ]:
# TODO: convert each tokenized document into bag-of-words counts with doc2bow.
corpus = ...

# TODO: inspect the number of documents and a few (token_id, count) pairs.
# TODO: map the first document's token IDs back to words.


### 4 — Choose the number of topics

`num_topics` is a modeling choice. Too few topics can merge distinct themes; too many can split a coherent theme into fragments. There is no universally correct number. In practice, compare several values, inspect topic coherence and redundancy, read representative documents, and use domain knowledge. For this worked example, we will fit 23 topics to match the NMF setting.

In [ ]:
# TODO: choose a number of topics for LDA. Try comparing values later.
num_topics = ...


### 5 — Train the LDA model

`LdaModel` learns topic-word and document-topic distributions from the corpus. `passes` controls how many full passes are made over the corpus during training. `random_state` fixes the random seed for reproducibility. We use the same 23-topic setting as the NMF example, while remembering the models represent topic weights differently.

In [ ]:
# TODO: build an LdaModel using corpus, id2word, and num_topics.
# Use passes=20 and set a random_state for reproducibility.
lda_model = ...


### 6 — Inspect topic-word distributions (topic loadings)

Each LDA topic is a probability distribution over vocabulary terms. Print the 15 highest-probability words for each topic. The listed numbers are probabilities for words within each topic; they are not document-topic scores.

In [ ]:
# TODO: print the top 15 terms for every topic using lda_model.print_topics().


In [ ]:
# TODO: inspect whether selected topics are associated with the Romance genre.
# Rank documents by their probability for a topic and examine their genres.


In [ ]:
# TODO: inspect whether selected LDA topics are associated with the Romance genre.
# Rank documents by probability for each topic and examine their genres.


### 7 — Inspect topic mixtures for documents (document scores)

For each document, LDA estimates a distribution over topics. `get_document_topics` returns `(topic_id, probability)` pairs. By default, very small probabilities may be omitted; set `minimum_probability=0` when you want all topics included. First inspect one synopsis and its topic distribution.

In [ ]:
# TODO: choose a document, display its synopsis, get its full topic distribution
# with minimum_probability=0, and print its five highest-probability topics.


### 8 — Discuss topic membership

What score is high enough to say a document “belongs” to a topic? There is no fixed universal cutoff. A topic may be the document’s strongest theme even when its probability is modest, especially when the document spreads probability across several topics. Consider ranking topics, comparing the top score with the next one, and reading the document before making a claim.

In [ ]:
# TODO: write a helper function that returns the n highest-probability topics
# for a bag-of-words document, then inspect a few documents.


### 9 — Create an interactive pyLDAvis visualization

pyLDAvis presents two complementary views: topic circles summarize prevalence and inter-topic distance, while the term panel shows terms relevant to a selected topic. Circle size reflects topic prevalence; distance between circles approximates topic dissimilarity in the projected view. The visualization is a diagnostic aid, not proof that a topic is valid.

If pyLDAvis is not installed in your environment, install it once with `%pip install pyLDAvis` and restart the kernel if prompted.

In [ ]:
%pip install pyLDAvis

In [ ]:
%pip install --upgrade pyLDAvis scikit-learn

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn.manifold._mds"
)

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvisualize

pyLDAvis.enable_notebook(local=True)

Prepare the visualization using the trained Gensim model, its bag-of-words corpus, and dictionary. Then display it in the notebook. Review the topics with the top words and representative documents from the previous steps.

In [ ]:
# TODO: prepare the pyLDAvis visualization from lda_model, corpus, and id2word.
# Then display it in the notebook.


**Final discussion.** Do topics that appear close in the map share similar words? Are any topics nearly indistinguishable or represented by only a few generic terms? Use the map to decide which topics deserve closer inspection, then return to their top words and high-scoring documents. Topic quality requires interpretation alongside the visualization.